# Voronoi Volume Estimation for Decoder Embeddings

This notebook loads a model (or a pre-saved decoder matrix), restricts to an optional token subset (including the PG19 validation chunks), and estimates Voronoi cell volumes inside either a radius-`R` ball or the convex hull of selected tokens using `voronoi_visualizer.voronoi_volume`.

In [ ]:
from pathlib import Path
import json
import torch
from voronoi_visualizer.model_factory import load_model
from voronoi_visualizer.voronoi_volume import estimate_voronoi_cell_volumes

DATA_DIR = Path('data')
RECTANGLE_BOUNDS_DIR = DATA_DIR / 'rects_inf'
VORONOI_RESULTS_DIR = DATA_DIR / 'voronoi_results'
FIGURE_DIR = Path('figures')
VORONOI_FIGURE_DIR = FIGURE_DIR / 'voronoi_volume_distributions'
CONVOLUTION_FIGURE_PATH = FIGURE_DIR / 'voronoi_convolution_histograms_grid.pdf'
PG19_CSV_PATH = Path('cramming/data/pg19_valid_1k_chunks.csv')
OUTPUT_JSON = VORONOI_RESULTS_DIR / 'voronoi_volumes.json'

print(torch.__version__)

In [ ]:
# ---- Configuration ----
# Either set `decoder_matrix_path` to a saved decoder matrix (shape [d, vocab] if tokens are columns,
# or [vocab, d] if tokens are rows), or leave as None to load a model and pull its lm_head.
decoder_matrix_path = None  # e.g., Path('models/llms-theory/my_decoder.pt')
tokens_on_columns = True     # set True if columns are tokens when using a saved matrix

models = [
    'EleutherAI/pythia-160m',
    'EleutherAI/pythia-410m',
    'EleutherAI/pythia-1b',
    'Qwen/Qwen2.5-0.5B',
    'Qwen/Qwen2.5-1.5B',
    'meta-llama/Llama-3.2-1B',
    'google/gemma-3-270m',
]

# Model loading (ignored if decoder_matrix_path is set)
# Local paths resolve against $LLM_VIS_MODEL_ROOT or models/llms-theory.
model_name = 'EleutherAI/pythia-160m'  # repo id or local dir under the roots above

# Estimation hyperparameters
region = "convex_hull"  # "ball" or "convex_hull"
radius = 10.0            # only used when region == "ball"
num_samples = 50_000_000
chunk_size_tokens = 2048   # reduce if you hit OOM
point_batch_size = 4096    # reduce if you hit OOM
seed = 0
device = "cuda" if torch.cuda.is_available() else "cpu"

# Convex hull sampling controls
convex_hull_simplex_size = 64  # tokens per sample when region == "convex_hull"

# Token subset controls. Provide IDs directly and/or token strings (strings require tokenizer).
subset_token_ids = []
subset_token_strings = []  # e.g., [" the", ".</w>"]

# Dataset-based token subset (only keep tokens that appear in PG19)
use_pg19_token_subset = False
pg19_csv_path = PG19_CSV_PATH
pg19_text_column = 'text'
pg19_max_rows = None  # set to an int to cap rows for faster debugging
pg19_tokenize_batch_size = 32

# Optional output path to persist results
output_json = OUTPUT_JSON

In [ ]:
# ---- Load decoder matrix ----
import numpy as np
tokenizer = None

if decoder_matrix_path is None:
    model = load_model(model_name)
    if hasattr(model, 'resolved_path') and model.resolved_path:
        print(f'Resolved local model path: {model.resolved_path}')
    tokenizer = model.tokenizer
    decoder_matrix = model.get_output_projection_matrix()  # shape [vocab, dim]
    tokens_on_columns = False
    print(f"Loaded model {model_name}; decoder matrix shape: {tuple(decoder_matrix.shape)}")
else:
    decoder_matrix = torch.load(decoder_matrix_path)
    print(f"Loaded decoder matrix from {decoder_matrix_path}; shape: {tuple(decoder_matrix.shape)}")

decoder_matrix = decoder_matrix.to(device)

# Build subset indices
subset_ids = list(subset_token_ids)
if subset_token_strings:
    if tokenizer is None:
        raise ValueError("subset_token_strings requires a tokenizer; load a model instead of raw matrix.")
    for tok in subset_token_strings:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid == tokenizer.unk_token_id or tid is None:
            print(f"Warning: token '{tok}' not found; skipping.")
            continue
        subset_ids.append(int(tid))

pg19_token_ids = None
if use_pg19_token_subset:
    if tokenizer is None:
        raise ValueError("PG19 token subset requires a tokenizer; load a model instead of raw matrix.")
    if not pg19_csv_path.exists():
        raise FileNotFoundError(f"PG19 CSV not found at {pg19_csv_path}")

    import csv
    import sys

    max_size = sys.maxsize
    while True:
        try:
            csv.field_size_limit(max_size)
            break
        except OverflowError:
            max_size = max_size // 10

    pg19_token_ids = set()
    batch = []
    row_count = 0
    with open(pg19_csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if pg19_max_rows is not None and row_count >= pg19_max_rows:
                break
            text = row.get(pg19_text_column)
            if text is None:
                continue
            batch.append(text)
            row_count += 1
            if len(batch) >= pg19_tokenize_batch_size:
                enc = tokenizer(batch, add_special_tokens=False, padding=False, truncation=False)
                for ids in enc["input_ids"]:
                    pg19_token_ids.update(ids)
                batch = []
        if batch:
            enc = tokenizer(batch, add_special_tokens=False, padding=False, truncation=False)
            for ids in enc["input_ids"]:
                pg19_token_ids.update(ids)

    print(f"PG19 unique tokens: {len(pg19_token_ids)}")

if pg19_token_ids is not None:
    if subset_ids:
        subset_ids = sorted(set(subset_ids).intersection(pg19_token_ids))
    else:
        subset_ids = sorted(pg19_token_ids)
else:
    subset_ids = sorted(set(subset_ids)) if subset_ids else None

print(f"Subset: {subset_ids if subset_ids is not None else 'all tokens'}")

In [ ]:
# ---- Estimate Voronoi volumes ----
# Load bounds from file if region is "rectangle", otherwise set to None
bounds_file = RECTANGLE_BOUNDS_DIR / f"{model_name.split('/')[-1]}.txt"
bounds_list = []
with open(bounds_file, "r") as f:
    for line in f:
        min_val, max_val = map(float, line.strip().split())
        bounds_list.append([min_val, max_val])
bounds = torch.tensor(bounds_list, dtype=decoder_matrix.dtype, device=device)
print(f"Loaded bounds from {bounds_file}; shape: {bounds.shape}")

result = estimate_voronoi_cell_volumes(
    decoder_matrix=decoder_matrix,
    region=region,
    radius=radius if region == "ball" else None,
    bounds=bounds,
    subset_indices=subset_ids,
    num_samples=num_samples,
    tokens_on_columns=tokens_on_columns,
    device=device,
    chunk_size_tokens=chunk_size_tokens,
    point_batch_size=point_batch_size,
    seed=seed,
    convex_hull_simplex_size=convex_hull_simplex_size,
)

print({k: result[k] for k in ["region", "radius", "dim", "num_samples", "region_volume"]})

In [ ]:
# ---- Inspect results ----
def token_str(tid: int) -> str:
    if tokenizer is None:
        return f"<id_{tid}>"
    try:
        return tokenizer.decode([tid])
    except Exception:
        return f"<id_{tid}>"

items = []
for tid, vol in result["volume_per_token"].items():
    items.append({
        "token_id": tid,
        "token": token_str(tid),
        "volume": vol,
        "fraction": result["proportion_per_token"][tid],
        "samples": result["counts"][tid],
    })

items_sorted = sorted(items, key=lambda x: x["volume"], reverse=False)
print(f"Top 10 by volume (out of {len(items_sorted)} tokens):")
count_0 = 0 
i = 0
while i < len(items_sorted) and items_sorted[i]["volume"] == 0.0:
    count_0 += 1
    i += 1

print(f"  (There are {count_0} tokens with zero volume), ratio {count_0/len(items_sorted):.6e}")
for entry in items_sorted[count_0:count_0+10]:
    print(f"  id {entry['token_id']:5d} | vol {entry['volume']:.6e} | frac {entry['fraction']:.6e} | samples {entry['samples']:7d} | token '{entry['token']}'")   

print(f"Top 10 by volume (out of {len(items_sorted)} tokens):")
for entry in items_sorted[-10:]:
    print(f"  id {entry['token_id']:5d} | vol {entry['volume']:.6e} | frac {entry['fraction']:.6e} | samples {entry['samples']:7d} | token '{entry['token']}'")   
 

In [ ]:
# ---- Distribution plots (log-log scatter + Pareto-style cumulative) ----
import matplotlib.pyplot as plt

vols = torch.tensor([v for v in result["volume_per_token"].values()], dtype=torch.float64)
props = torch.tensor([v for v in result["proportion_per_token"].values()], dtype=torch.float64)
finite_pos = torch.isfinite(vols) & (vols > 0)
if finite_pos.sum() == 0:
    print('No positive finite volumes to plot.')
else:
    sorted_vols, _ = torch.sort(vols[finite_pos], descending=True)
    ranks = torch.arange(1, len(sorted_vols) + 1)
    plt.figure(figsize=(6,4))
    plt.scatter(ranks.cpu().numpy(), sorted_vols.cpu().numpy(), s=8, alpha=0.6, color='steelblue', edgecolors='none')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('token rank (sorted by volume)')
    plt.ylabel('Voronoi volume')
    plt.title('Log-log rank vs Voronoi volume')
    plt.grid(alpha=0.3, which='both')
    plt.show()

    sorted_props, _ = torch.sort(props, descending=True)
    cum = torch.cumsum(sorted_props, dim=0)
    frac_tokens = torch.linspace(0, 1, steps=len(sorted_props))
    plt.figure(figsize=(6,4))
    plt.plot(frac_tokens.cpu().numpy(), cum.cpu().numpy(), label='cumulative fraction of volume')
    plt.axhline(0.8, color='red', linestyle='--', alpha=0.6, label='80% volume')
    plt.xlabel('fraction of tokens (sorted by volume)')
    plt.ylabel('cumulative volume fraction')
    plt.ylim(0,1.05)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title('Pareto-style cumulative coverage')
    plt.show()

    # Rough Pareto point: min number of tokens to reach 80% of mass
    pareto_idx = torch.nonzero(cum >= 0.8, as_tuple=False)
    if pareto_idx.numel() > 0:
        k = pareto_idx[0].item() + 1
        print(f'Tokens needed for 80% of volume: {k} ({k/len(sorted_props):.2%} of tokens)')
    else:
        print('Never reached 80% (unexpected)')

In [ ]:
# ---- Save (optional) ----
if output_json is not None:
    output_json = Path(output_json)
    payload = {
        **result,
        "token_strings": {tid: token_str(tid) for tid in result["volume_per_token"].keys()},
    }
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved to {output_json}")

## Cells volumes distributions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def plot_voronoi_volume_distribution(model_name: str, csv_path: Path, output_dir: Path):
    # Load CSV
    df = pd.read_csv(csv_path)
    props = df['proportion'].values

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Log-log scatter (rank vs proportion)
    sorted_props = np.sort(props)[::-1]
    ranks = np.arange(1, len(sorted_props) + 1)
    positive = sorted_props > 0
    ax1.scatter(
        ranks[positive],
        sorted_props[positive],
        s=8,
        alpha=0.6,
        color='steelblue',
        edgecolors='none'
    )
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlabel('token rank (sorted by proportion)')
    ax1.set_ylabel('proportion of volume')
    ax1.set_title('Log-log rank vs proportion')
    ax1.grid(alpha=0.3, which='both')

    # Pareto plot
    cum = np.cumsum(sorted_props)
    frac_tokens = np.linspace(0, 1, len(sorted_props))

    ax2.plot(frac_tokens, cum, label='cumulative fraction of volume')
    ax2.axhline(0.8, color='red', linestyle='--', alpha=0.6, label='80% volume')
    ax2.set_xlabel('fraction of tokens (sorted by proportion)')
    ax2.set_ylabel('cumulative proportion')
    ax2.set_ylim(0, 1.05)
    ax2.legend()
    ax2.grid(alpha=0.3)
    ax2.set_title('Pareto-style cumulative coverage')

    # Compute Pareto point
    pareto_idx = np.where(cum >= 0.8)[0]
    if len(pareto_idx) > 0:
        k = pareto_idx[0] + 1
        print(f'Tokens needed for 80% of volume: {k} ({k/len(sorted_props):.2%} of tokens)')
    plt.tight_layout()
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{model_name.replace('/', '_')}.svg"
    plt.savefig(output_path, format='svg', dpi=150)
    print(f"Saved figure to {output_path}")
    plt.close()

models = [
'EleutherAI/pythia-160m',
'EleutherAI/pythia-410m',
'EleutherAI/pythia-1b',
'EleutherAI/pythia-1.4b',
'EleutherAI/pythia-2.8b',
'Qwen/Qwen2.5-0.5B',
'Qwen/Qwen2.5-1.5B',
'meta-llama/Llama-3.2-1B',
'meta-llama/Llama-3.2-3B',
'meta-llama/Llama-3.1-8B',
'google/gemma-3-1b-pt',
'google/gemma-3-270m',
]

for model_name in models:
    print(f"\n{'='*60}")
    print(f"Processing: {model_name}")
    print(f"{'='*60}")
    csv_path = VORONOI_RESULTS_DIR / f"{model_name.replace('/', '_')}.csv"
    output_dir = VORONOI_FIGURE_DIR
    plot_voronoi_volume_distribution(model_name, csv_path, output_dir)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import numpy as np

def sample_ranked_points_log(
    sorted_props: np.ndarray,
    after_rank: int = 100,
    n_after: int = 3000,
    base: float = 10.0,
):
    """
    Keep all points up to `after_rank`, then sample `n_after` points from the tail
    with ranks spaced uniformly in log(rank). This avoids threshold artifacts on log-log plots.

    Inputs:
      sorted_props: 1D array, assumed sorted descending (proportions)
      after_rank:   keep ranks 1..after_rank (after removing zeros)
      n_after:      number of tail samples
      base:         log base for spacing (10 is standard for decades)

    Returns:
      ranks (1-based), props
    """
    # filter non-positive (log scale friendly)
    positive = sorted_props > 0
    props_pos = sorted_props[positive]
    N = len(props_pos)

    if N == 0:
        return np.array([], dtype=int), np.array([], dtype=sorted_props.dtype)

    ranks = np.arange(1, N + 1, dtype=int)

    if N <= after_rank or n_after <= 0:
        return ranks[:after_rank], props_pos[:after_rank]

    # head (dense)
    head_ranks = ranks[:after_rank]
    head_props = props_pos[:after_rank]

    # tail: log-spaced ranks in [after_rank+1, N]
    lo = after_rank + 1
    hi = N

    if (hi - lo + 1) <= n_after:
        tail_ranks = ranks[after_rank:]
        tail_props = props_pos[after_rank:]
    else:
        # sample ranks directly (not indices), then convert to indices
        # logspace gives floats; round + unique removes duplicates
        tail_ranks = np.logspace(
            np.log(lo) / np.log(base),
            np.log(hi) / np.log(base),
            num=n_after,
            base=base,
        )
        tail_ranks = np.unique(np.rint(tail_ranks).astype(int))
        tail_ranks = np.clip(tail_ranks, lo, hi)

        # map ranks -> indices
        tail_idx = tail_ranks - 1
        tail_props = props_pos[tail_idx]

    ranks_out = np.concatenate([head_ranks, tail_ranks])
    props_out = np.concatenate([head_props, tail_props])
    return ranks_out, props_out


def plot_overlay_loglog(
    models,
    csv_dir: Path,
    output_path: Path,
    color_map: dict,
    after_rank: int = 100,
    n_after: int = 3000,
):
    plt.figure(figsize=(6, 4))

    for model_name in models:
        csv_path = csv_dir / f"{model_name.replace('/', '_')}.csv"
        if not csv_path.exists():
            print(f"Missing CSV: {csv_path}")
            continue

        df = pd.read_csv(csv_path)
        props = df["proportion"].values
        sorted_props = np.sort(props)[::-1]

        ranks, subsampled = sample_ranked_points_log(
            sorted_props, after_rank=after_rank, n_after=n_after
        )
        print(f"Model {model_name}: plotting {len(subsampled)} points")
        color = color_map.get(model_name, None)
        plt.scatter(
            ranks,
            subsampled,
            s=8,
            alpha=1,
            color=color,
            edgecolors="none",
            label=model_name.split("/")[-1].capitalize(),
        )

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Token rank (sorted by decreasing proportion)")
    plt.ylabel("Proportion of samples in token cell")
    # plt.title("Log-log rank vs proportion (overlay)")
    plt.grid(alpha=0.1, which="both")
    plt.legend(fontsize=10, frameon=False)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')

    plt.tight_layout()
    plt.savefig(output_path, format="pdf", dpi=150)
    print(f"Saved figure to {output_path}")
    plt.show()
    plt.close()

models = [
    "EleutherAI/pythia-160m",
    "EleutherAI/pythia-410m",
    "EleutherAI/pythia-1b",
    "Qwen/Qwen2.5-0.5B",
    "Qwen/Qwen2.5-1.5B",
    "meta-llama/Llama-3.2-1B",
    "google/gemma-3-270m",
]
# Same hue per family, vary lightness within family
color_map = {
    # Pythia (blue hues)
    "EleutherAI/pythia-160m": "#87adf0",
    "EleutherAI/pythia-410m": "#45a1e2",
    "EleutherAI/pythia-1b":   "#215E8C",

    # Qwen (green hues)
    "Qwen/Qwen2.5-0.5B": "#4aac7b",
    "Qwen/Qwen2.5-1.5B": "#177517",

    # Llama (orange hues)
    "meta-llama/Llama-3.2-1B": "#ec8333",

    # Gemma (purple hues)
    "google/gemma-3-270m":  "#934fd3",
}
plot_overlay_loglog(
    models=models,
    csv_dir=VORONOI_RESULTS_DIR,
    output_path=VORONOI_FIGURE_DIR / "overlay_loglog.pdf",
    color_map=color_map,
    after_rank=100,
    n_after=500,
)

### Plot histogram volumes 

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional: set to "kde" if seaborn is available, otherwise "hist"
plot_kind = "hist"

csv_dir = VORONOI_RESULTS_DIR
models = [
    "EleutherAI/pythia-160m",
    "EleutherAI/pythia-410m",
    "EleutherAI/pythia-1b",
    "Qwen/Qwen2.5-0.5B",
    "Qwen/Qwen2.5-1.5B",
    "meta-llama/Llama-3.2-1B",
    "google/gemma-3-270m",
]

# Load data per model
model_props = {}
for m in models:
    csv_path = csv_dir / f"{m.replace('/', '_')}.csv"
    if not csv_path.exists():
        model_props[m] = None
        continue
    df = pd.read_csv(csv_path)
    props = df["proportion"].to_numpy()
    props = props[np.isfinite(props) & (props > 0)]
    model_props[m] = props if props.size else None

# Global bins for comparable histograms
all_props = np.concatenate([p for p in model_props.values() if p is not None])
if all_props.size == 0:
    raise RuntimeError("No valid proportion data found.")
bins = np.logspace(np.log10(all_props.min()), np.log10(all_props.max()), 60)

# Subplot grid
n = len(models)
cols = 3
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.2, rows * 3.2))
axes = np.array(axes).reshape(-1)

# Optional KDE setup
if plot_kind == "kde":
    try:
        import seaborn as sns
    except Exception:
        plot_kind = "hist"

for ax, m in zip(axes, models):
    props = model_props[m]
    ax.set_title(m.split("/")[-1])

    if props is None:
        ax.text(0.5, 0.5, "no data", ha="center", va="center")
        ax.set_axis_off()
        continue

    if plot_kind == "kde":
        sns.kdeplot(props, bw_adjust=0.6, fill=True, color="slateblue", ax=ax)
    else:
        ax.hist(props, bins=bins, color="slateblue", alpha=0.85, edgecolor="white", linewidth=0.4)

    ax.set_xscale("log")
    ax.set_xlabel("proportion")
    ax.set_ylabel("density" if plot_kind == "kde" else "count")
    ax.grid(alpha=0.3, which="both")

# Hide unused axes
for ax in axes[n:]:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

### Convolution 

In [ ]:
from __future__ import annotations

from math import exp, log
from typing import List, Tuple, Dict, Optional


Dist = List[Tuple[float, float]]  # [(value, prob)], sorted by value


def _normalize(d: Dist) -> Dist:
    s = sum(p for _, p in d)
    if s <= 0:
        raise ValueError("Total probability mass must be > 0.")
    out = [(v, p / s) for v, p in d if p > 0]
    out.sort(key=lambda t: t[0])
    return out

def _median(d: Dist) -> float:
    """d must be sorted by value and normalized."""
    c = 0.0
    for v, p in d:
        c += p
        if c >= 0.5:
            return v
    return d[-1][0]  # numerical fallback

def _compress_to_n(d: Dist, n: int) -> Dist:
    """
    Compress a sorted discrete distribution to at most n support points
    by grouping consecutive points into ~equal-probability bins.

    Representative value per bin:
      - weighted geometric mean if all values in bin are > 0
      - otherwise weighted arithmetic mean
    """
    if n <= 0:
        raise ValueError("n must be >= 1.")
    if len(d) <= n:
        return d

    target = 1.0 / n
    bins: List[List[Tuple[float, float]]] = []
    cur: List[Tuple[float, float]] = []
    mass = 0.0

    for v, p in d:
        cur.append((v, p))
        mass += p
        if mass >= target and len(bins) < n - 1:
            bins.append(cur)
            cur = []
            mass = 0.0

    if cur:
        bins.append(cur)

    out: Dist = []
    for b in bins:
        m = sum(p for _, p in b)
        if m <= 0:
            continue
        all_pos = all(v > 0 for v, _ in b)
        if all_pos:
            # weighted geometric mean: exp(sum w_i log v_i)
            lv = 0.0
            for v, p in b:
                lv += (p / m) * log(v)
            rep = exp(lv)
        else:
            # weighted arithmetic mean
            rep = sum(v * p for v, p in b) / m
        out.append((rep, m))

    out.sort(key=lambda t: t[0])
    return out


def _mul_convolve(dk: Dist, d: Dist) -> Dist:
    """Exact (dense) multiplication-convolution: support of dk * support of d."""
    acc: Dict[float, float] = {}
    for x, px in dk:
        for y, py in d:
            v = x * y
            acc[v] = acc.get(v, 0.0) + px * py
    out = [(v, p) for v, p in acc.items() if p > 0]
    out.sort(key=lambda t: t[0])
    return out


def f(n: int, d: Dist, epsilon: float, max_k: int = 10000) -> int:
    """
    Inputs
      - n: max support size used to represent each intermediate distribution (compression budget)
      - d: base discrete distribution of X (ordered list of [value, prob])
      - epsilon: stopping threshold

    Procedure
      For k = 1, 2, ...
        compute d_k = distribution of product X_1 * ... * X_k (iid X_i ~ d),
        represent d_k with at most n points (compression),
        stop when median(d_k) < epsilon and return k.

    Notes
      - This computes the product distribution exactly at each step, then compresses to <= n points.
      - If the stopping condition never happens within max_k, raises RuntimeError.
    """
    if epsilon != epsilon:
        raise ValueError("epsilon must be a real number (not NaN).")
    if max_k < 1:
        raise ValueError("max_k must be >= 1.")

    d0 = _normalize([(float(v), float(p)) for v, p in d])
    dk = d0  # k=1 distribution

    # k = 1 check
    dk = _compress_to_n(dk, n)
    if _median(dk) < epsilon:
        return 1

    for k in range(2, max_k + 1):
        dk = _mul_convolve(dk, d0)
        dk = _normalize(dk)
        dk = _compress_to_n(dk, n)
        if _median(dk) < epsilon:
            return k

    raise RuntimeError(f"Stopping condition not met for k <= {max_k}.")




# Usage example:
# d = [(0.5, 0.6), (2.0, 0.4)]   # X in {0.5, 2} with given probs
# k = f(n=200, d=d, epsilon=1e-3)
# print(k)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def _dist_from_proportions(props, weight_mode="uniform"):
    props = np.asarray(props, dtype=float)
    props = props[props > 0]
    if props.size == 0:
        raise ValueError("No positive proportions.")
    if weight_mode == "uniform":
        p = 1.0 / props.size
        d = [(float(v), p) for v in props]
    elif weight_mode == "volume":
        d = [(float(v), float(v)) for v in props]
    else:
        raise ValueError("weight_mode must be 'uniform' or 'volume'")
    return _normalize(d), int(props.size)

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Assumes these exist from your code:
# _normalize, _compress_to_n, _mul_convolve, _dist_from_proportions

def plot_convolution_histograms_grid(
    models,
    csv_dir: Path,
    max_k: int = 4,
    n: int = 2000,
    weight_mode: str = "uniform",
    num_bins: int = 70,
):
    cols = 3
    rows = math.ceil(len(models) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.4, rows * 3.4))
    axes = np.array(axes).reshape(-1)

    for ax, model_name in zip(axes, models):
        csv_path = csv_dir / f"{model_name.replace('/', '_')}.csv"
        if not csv_path.exists():
            ax.text(0.5, 0.5, "missing CSV", ha="center", va="center")
            ax.set_axis_off()
            continue

        df = pd.read_csv(csv_path)
        d0, _ = _dist_from_proportions(df["proportion"].values, weight_mode=weight_mode)

        dks = {}
        dk = _compress_to_n(d0, n)
        dks[1] = dk
        for k in range(2, max_k + 1):
            dk = _mul_convolve(dk, d0)
            dk = _normalize(dk)
            dk = _compress_to_n(dk, n)
            dks[k] = dk

        all_vals = np.concatenate([
            np.array([v for v, _ in dks[k]], dtype=float) for k in dks
        ])
        all_vals = all_vals[np.isfinite(all_vals) & (all_vals > 0)]
        if all_vals.size == 0:
            ax.text(0.5, 0.5, "no data", ha="center", va="center")
            ax.set_axis_off()
            continue

        log_min = np.log10(all_vals.min())
        log_max = np.log10(all_vals.max())
        bins = np.linspace(log_min, log_max, num_bins + 1)

        for k in range(1, max_k + 1):
            vals = np.array([v for v, _ in dks[k]], dtype=float)
            weights = np.array([p for _, p in dks[k]], dtype=float)
            mask = np.isfinite(vals) & (vals > 0) & np.isfinite(weights)
            hist, edges = np.histogram(np.log10(vals[mask]), bins=bins, weights=weights[mask])
            if hist.sum() > 0:
                hist = hist / hist.sum()
            centers = (edges[:-1] + edges[1:]) / 2.0
            ax.step(10 ** centers, hist, where="mid", label=f"k={k}", alpha=0.9)

        ax.set_xscale("log")
        ax.set_xlabel("volume (proportion)")
        ax.set_ylabel("prob mass")
        ax.grid(alpha=0.2, which="both")
        ax.set_title(model_name.split("/")[-1])
        ax.legend(frameon=False, fontsize=8)

    for ax in axes[len(models):]:
        ax.set_axis_off()

    plt.tight_layout()
    plt.show()

# Example
models = [
    "EleutherAI/pythia-160m",
    "EleutherAI/pythia-410m",
    "EleutherAI/pythia-1b",
    "Qwen/Qwen2.5-0.5B",
    "Qwen/Qwen2.5-1.5B",
    "meta-llama/Llama-3.2-1B",
    "google/gemma-3-270m",
]

plot_convolution_histograms_grid(
    models=models,
    csv_dir=VORONOI_RESULTS_DIR,
    max_k=2,
    n=2000,
    weight_mode="uniform",
    num_bins=70,
)

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Assumes these exist from your code:
# _normalize, _compress_to_n, _mul_convolve, _dist_from_proportions

def plot_convolution_histograms_grid(
    models,
    csv_dir: Path,
    max_k: int = 4,
    n: int = 2000,
    weight_mode: str = "uniform",
    num_bins: int = 70,
):
    cols = 3
    rows = math.ceil(len(models) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.4, rows * 3.4))
    axes = np.array(axes).reshape(-1)

    for ax, model_name in zip(axes, models):
        csv_path = csv_dir / f"{model_name.replace('/', '_')}.csv"
        if not csv_path.exists():
            ax.text(0.5, 0.5, "missing CSV", ha="center", va="center")
            ax.set_axis_off()
            continue

        df = pd.read_csv(csv_path)
        d0, _ = _dist_from_proportions(df["proportion"].values, weight_mode=weight_mode)

        dks = {}
        dk = _compress_to_n(d0, n)
        dks[1] = dk
        for k in range(2, max_k + 1):
            dk = _mul_convolve(dk, d0)
            dk = _normalize(dk)
            dk = _compress_to_n(dk, n)
            dks[k] = dk

        all_vals = np.concatenate([
            np.array([v for v, _ in dks[k]], dtype=float) for k in dks
        ])
        all_vals = all_vals[np.isfinite(all_vals) & (all_vals > 0)]
        if all_vals.size == 0:
            ax.text(0.5, 0.5, "no data", ha="center", va="center")
            ax.set_axis_off()
            continue

        log_min = np.log10(all_vals.min())
        log_max = np.log10(all_vals.max())
        bins = np.linspace(log_min, log_max, num_bins + 1)

        for k in range(1, max_k + 1):
            vals = np.array([v for v, _ in dks[k]], dtype=float)
            weights = np.array([p for _, p in dks[k]], dtype=float)
            mask = np.isfinite(vals) & (vals > 0) & np.isfinite(weights)
            hist, edges = np.histogram(np.log10(vals[mask]), bins=bins, weights=weights[mask])
            if hist.sum() > 0:
                hist = hist / hist.sum()
            centers = (edges[:-1] + edges[1:]) / 2.0
            ax.step(10 ** centers, hist, where="mid", label=f"k={k}", alpha=0.9)

        ax.set_xscale("log")
        ax.set_xlabel("volume (proportion)")
        ax.set_ylabel("prob mass")
        ax.grid(alpha=0.2, which="both")
        ax.set_title(model_name.split("/")[-1])
        ax.legend(frameon=False, fontsize=8)

    for ax in axes[len(models):]:
        ax.set_axis_off()

    CONVOLUTION_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(CONVOLUTION_FIGURE_PATH, format="pdf", dpi=150)
    plt.tight_layout()
    plt.show()

# Example
models = [
    "EleutherAI/pythia-160m",
    "EleutherAI/pythia-410m",
    "EleutherAI/pythia-1b",
    "Qwen/Qwen2.5-0.5B",
    "Qwen/Qwen2.5-1.5B",
    "meta-llama/Llama-3.2-1B",
    "google/gemma-3-270m",
]

plot_convolution_histograms_grid(
    models=models,
    csv_dir=VORONOI_RESULTS_DIR,
    max_k=4,
    n=2000,
    weight_mode="uniform",
    num_bins=70,
)